## IMPORT DATA EVENTS.CSV KE NEO4J

Koneksi ke Neo4j

In [20]:
import os
import pandas as pd
from getpass import getpass
from neo4j import GraphDatabase

# 1. Konfigurasi koneksi Neo4j
bolt_uri = "neo4j://127.0.0.1:7687"
neo4j_user = "neo4j"

# 2. Meminta password dari pengguna 
neo4j_password = getpass("Masukkan password untuk instance GraphSada: ")

# 3. Cek koneksi ke Neo4j
with GraphDatabase.driver(
    bolt_uri,
    auth=(neo4j_user, neo4j_password)
) as driver:
    driver.verify_connectivity()
    print("Koneksi ke Neo4j berhasil.")

Koneksi ke Neo4j berhasil.


Mendefinisikan Fungsi Import

In [21]:
def create_constraints(session):
    """Membuat constraint agar ID Visitor dan item bersifat unik."""
    session.run("""
        CREATE CONSTRAINT visitor_id_unique IF NOT EXISTS
        FOR (v:Visitor)
        REQUIRE v.visitorid IS UNIQUE
    """)

    session.run("""
        CREATE CONSTRAINT item_id_unique IF NOT EXISTS
        FOR (i:item)
        REQUIRE i.itemid IS UNIQUE
    """)


def import_batch(tx, records):
    """Mengimpor sekumpulan baris interaksi ke Neo4j."""

    cypher_query = """
    UNWIND $batch AS row

    MERGE (v:Visitor {visitorid: row.visitorid})
    MERGE (i:item {itemid: row.itemid})

    FOREACH (_ IN CASE
        WHEN row.event = 'view' THEN [1]
        ELSE []
    END |
        MERGE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)
    )

    FOREACH (_ IN CASE
        WHEN row.event = 'addtocart' THEN [1]
        ELSE []
    END |
        MERGE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)
    )

    FOREACH (_ IN CASE
        WHEN row.event = 'transaction' THEN [1]
        ELSE []
    END |
        MERGE (v)-[:BOUGHT {
            timestamp: row.timestamp,
            transactionid: row.transactionid
        }]->(i)
    )

    RETURN count(row) AS processed_rows
    """

    result = tx.run(cypher_query, batch=records)
    return result.single()["processed_rows"]


Import Data

In [11]:
from tqdm import tqdm

# 1. Konfigurasi file CSV dan ukuran batch
csv_file_path = "ecommerce-dataset\events.csv"
batch_size = 1000


# 2. Cek apakah file CSV ada
if not os.path.exists(csv_file_path):
    raise FileNotFoundError(
        f"File tidak ditemukan: {csv_file_path}"
    )


# 3. Impor Data ke Neo4j
with driver.session(database="neo4j") as session:

    create_constraints(session)

    print(f"Membaca file data dari: {csv_file_path}")

    chunks = pd.read_csv(
        csv_file_path,
        chunksize=batch_size
    )

    total_rows = 0

    for chunk in tqdm(chunks, desc="Mengimpor batch data"):

        # Pastikan nama kolom sesuai dengan dataset
        required_columns = {
            "timestamp",
            "visitorid",
            "event",
            "itemid",
            "transactionid"
        }

        missing_columns = required_columns - set(chunk.columns)

        if missing_columns:
            raise ValueError(
                f"Kolom tidak ditemukan: {missing_columns}"
            )

        # Bersihkan data
        chunk["event"] = (
            chunk["event"]
            .astype(str)
            .str.strip()
            .str.lower()
        )

        chunk["transactionid"] = (
            chunk["transactionid"]
            .fillna("")
            .astype(str)
        )

        # Pertahankan hanya jenis interaksi yang dipakai
        chunk = chunk[
            chunk["event"].isin(
                ["view", "addtocart", "transaction"]
            )
        ]

        records = chunk.to_dict(orient="records")

        if records:
            processed = session.execute_write(
                import_batch,
                records
            )
            total_rows += processed

    print("Proses impor data selesai.")
    print(f"Jumlah baris yang diproses: {total_rows}")


C:\Users\ANNAS\AppData\Local\Temp\ipykernel_7812\3546760525.py:16: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database="neo4j") as session:


Membaca file data dari: ecommerce-dataset\events.csv


Mengimpor batch data: 2757it [04:56,  9.31it/s]

Proses impor data selesai.
Jumlah baris yang diproses: 2756101


Melihat Rentang Waktu yang ada pada data "events.csv"

In [24]:
df_time = pd.read_csv(
    csv_file_path,
    usecols=["timestamp"]
)

timestamp = pd.to_datetime(
    df_time["timestamp"],
    unit="ms",
    errors="coerce"
)

print("Aktivitas paling awal :", timestamp.min())
print("Aktivitas paling akhir:", timestamp.max())
print("Jumlah baris           :", len(df_time))

Aktivitas paling awal : 2015-05-03 03:00:04.384000
Aktivitas paling akhir: 2015-09-18 02:59:47.788000
Jumlah baris           : 2756101


### 1. Membagi Data menjadi 2 fold (P1 dan P2) dengan rentang waktu yang telah ditentukan 

2. Simpan file hasil pembagian data
- P1 --> 3 Juni 2015 - 22 Agustus 2015
- P2 --> 23 Agustus 2015 - 12 September 2015

In [25]:
from pathlib import Path

# File sumber
csv_file_path = Path("ecommerce-dataset\events.csv")

# File hasil pembagian periode
p1_file = Path("events_P1.csv")
p2_file = Path("events_P2.csv")

# Batas periode (waktu akhir tidak termasuk)
p1_start = pd.Timestamp("2015-06-03 00:00:00")
p2_start = pd.Timestamp("2015-08-23 00:00:00")
data_end = pd.Timestamp("2015-09-12 00:00:00")

# Ukuran pembacaan per batch
chunksize = 200_000

# Hapus file hasil sebelumnya agar tidak tercampur
for output_file in [p1_file, p2_file]:
    if output_file.exists():
        output_file.unlink()

p1_rows = 0
p2_rows = 0

for chunk in pd.read_csv(csv_file_path, chunksize=chunksize):
    # Ubah timestamp milidetik menjadi datetime
    dates = pd.to_datetime(
        chunk["timestamp"],
        unit="ms",
        errors="coerce"
    )

    # Pilih data untuk P1 dan P2
    mask_p1 = (dates >= p1_start) & (dates < p2_start)
    mask_p2 = (dates >= p2_start) & (dates < data_end)

    chunk_p1 = chunk.loc[mask_p1]
    chunk_p2 = chunk.loc[mask_p2]

    # Simpan secara bertahap; header hanya ditulis pertama kali
    if not chunk_p1.empty:
        chunk_p1.to_csv(
            p1_file,
            mode="a",
            index=False,
            header=(p1_rows == 0)
        )
        p1_rows += len(chunk_p1)

    if not chunk_p2.empty:
        chunk_p2.to_csv(
            p2_file,
            mode="a",
            index=False,
            header=(p2_rows == 0)
        )
        p2_rows += len(chunk_p2)

print("Pembagian periode selesai.")
print(f"Jumlah baris P1: {p1_rows:,}")
print(f"Jumlah baris P2: {p2_rows:,}")
print(f"File P1: {p1_file.resolve()}")
print(f"File P2: {p2_file.resolve()}")
print(f"Total baris terpilih: {p1_rows + p2_rows:,}")


Pembagian periode selesai.
Jumlah baris P1: 1,655,156
Jumlah baris P2: 367,315
File P1: D:\ANNAS\DOKUMEN\TUGAS KULIAH\SEMESTER 7\GRAF SAINS DATA\UTS\events_P1.csv
File P2: D:\ANNAS\DOKUMEN\TUGAS KULIAH\SEMESTER 7\GRAF SAINS DATA\UTS\events_P2.csv
Total baris terpilih: 2,022,471


Validasi Pembagian Data

In [26]:

for file_path in [p1_file, p2_file]:
    if not file_path.exists():
        print(f"{file_path.name}: file tidak terbentuk")
        continue

    df_check = pd.read_csv(
        file_path,
        usecols=["timestamp"]
    )

    dates = pd.to_datetime(
        df_check["timestamp"],
        unit="ms",
        errors="coerce"
    )

    print(f"\nFile: {file_path.name}")
    print("Jumlah baris :", len(df_check))
    print("Tanggal awal :", dates.min())
    print("Tanggal akhir:", dates.max())



File: events_P1.csv
Jumlah baris : 1655156
Tanggal awal : 2015-06-03 00:00:03.487000
Tanggal akhir: 2015-08-22 23:59:53.877000

File: events_P2.csv
Jumlah baris : 367315
Tanggal awal : 2015-08-23 00:00:01.846000
Tanggal akhir: 2015-09-11 23:59:51.336000


### Uraian Jenis Simpul dan Jumlah masing-masing simpul

In [ ]:
query_komposisi = """
MATCH (n)
UNWIND labels(n) AS jenis_simpul
RETURN
    jenis_simpul,
    count(DISTINCT n) AS jumlah_simpul
ORDER BY jenis_simpul
"""

with driver.session(database="neo4j") as session:
    hasil = session.run(query_komposisi)
    df_komposisi = pd.DataFrame(
        [record.data() for record in hasil]
    )

display(df_komposisi)

C:\Users\ANNAS\AppData\Local\Temp\ipykernel_7812\2632427478.py:10: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database="neo4j") as session:


,jenis_simpul,jumlah_simpul
0,Visitor,1407580
1,item,235061
